In [ ]:
from pynq import Overlay, allocate
import numpy as np
import cv2

# Load FPGA bitstream
overlay = Overlay("design.bit")

# DMA IP (check your IP name in Vivado)
dma = overlay.axi_dma_0

# Image size (adjust based on your model)
HEIGHT = 32
WIDTH = 132
CHANNELS_IN = 3
CHANNELS_OUT = 31

# Allocate memory buffers
input_buffer = allocate(shape=(HEIGHT, WIDTH, CHANNELS_IN), dtype=np.float32)
output_buffer = allocate(shape=(HEIGHT, WIDTH, CHANNELS_OUT), dtype=np.float32)


def preprocess_image(img_path):
    img = cv2.imread(img_path)
    img = cv2.resize(img, (WIDTH, HEIGHT))
    img = img.astype(np.float32) / 255.0
    return img


def run_fpga_inference(img_path):
    # Load image
    img = preprocess_image(img_path)

    # Copy to input buffer
    np.copyto(input_buffer, img)

    # Start DMA transfer
    dma.sendchannel.transfer(input_buffer)
    dma.recvchannel.transfer(output_buffer)

    dma.sendchannel.wait()
    dma.recvchannel.wait()

    return np.array(output_buffer)


# Run test
output = run_fpga_inference("test.jpg")

print("Output shape:", output.shape)